# Lab 02 — Sensor Time Series with Pure Python
**Data Wrangling Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Load CSV time series with csv.DictReader and datetime.date
2. Aggregate monthly means with defaultdict(list)
3. Find hottest/coldest days and consecutive cold streaks
4. Flag anomalies with |z| > 2

## Datasets (this folder)
- `daily-min-temperatures.csv` — auto-download from `https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-02-sensor-time-series-pure-python/lab-02-sensor-time-series-pure-python.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`daily-min-temperatures.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-02-sensor-time-series-pure-python"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-02-sensor-time-series-pure-python/bundle/dataset.zip"
NEED = ["daily-min-temperatures.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Data Wrangling Track: Daily Temperatures, No Pandas

> **Scenario:** Ops hands you `daily-min-temperatures.csv` — 3650 rows of Melbourne daily minimum temperatures (Date, Temp, 1981–1990). Answer: *hottest day, coldest day, monthly averages, and the longest 3-day+ cold streak* — using only the Python standard library (`csv`, `datetime`, `statistics`).
>
> **You will learn:** `csv.DictReader`, date parsing, list/dict aggregation, `statistics.mean`, simple anomaly detection (|z| > 2).
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** Python 3.8+ only (no pandas). **Env:** 🟢 Colab only.

### Time-series mental map

Why it matters: sensor and ops data almost always arrives as a long date/value table. Once you can translate "column of temps," "group by month," and "running streak" into list/dict shapes, every section of this lab is a small, checkable transformation of the same 3650-row series — no pandas required.

| Spreadsheet idea | Python idea | Example |
|---|---|---|
| One column of temps | `list[float]` | `[20.7, 21.8, …]` |
| Group by month | `dict[(year, month) → list]` | `{(1981, 1): [13.9, …]}` |
| Row with date + value | `tuple` or `dict` | `("1981-01-01", 20.7)` |
| Running streak counter | keep previous date + counter | cold-run length |
| Excel filter (`Temp > 25`) | list comprehension with `if` | `[(d, t) for d, t in temps if t > 25]` |
| `=INDEX(MATCH(MAX(range)))` | `max(seq, key=…)` | hottest single row |

---

## 1. Load the data (local first, Colab fallback)

Why it matters for analysts: parsing dates and floats at load time is the cheapest place to catch a broken export. This loader prefers a local `datasets/` copy, falls back to a raw GitHub URL on Colab, and converts every row into a `(datetime.date, float)` pair so later sections never touch strings again.

In [ ]:
import csv, os, statistics
from datetime import datetime, date
from collections import defaultdict

def load_temps():
    """Load (date, temp) pairs. Local file first, else raw GitHub (Colab)."""
    local = "datasets/daily-min-temperatures.csv"
    if os.path.exists(local):
        path = local
    else:
        import urllib.request
        url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
        path = "daily-min-temperatures.csv"
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)

    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            d = datetime.strptime(r["Date"], "%Y-%m-%d").date()
            rows.append((d, float(r["Temp"])))
    return rows

temps = load_temps()
print(f"{len(temps)} rows, first={temps[0]}, last={temps[-1]}")
# Expect: 3650 rows, first=('1981-01-01', 20.7), last=('1990-12-31', 13.6)


> Why stdlib? Interviewers and constrained environments often ban pandas. `csv` + `datetime` is enough for a clean single-file series — and forcing yourself through `strptime` teaches exactly which format string your dates need.

**What to notice:**
- The file loads `3650` rows spanning `1981-01-01` to `1990-12-31` — ten full years of daily minima (365 × 10), so month counts in Section 3 should land on `120` and any missing-day gap would show up as a shorter streak in Section 5.
- First temp `20.7` and last temp `13.6` are real values from the CSV, already `float`, so no string comparison like `"9.5" > "10.2"` can sneak into later maths.
- Each element is a 2-tuple `(date, temp)`; unpacking with `for d, t in temps` is the loop shape used everywhere below.
- The local-path check runs first, so re-running the notebook offline does not re-download once the file exists.

> **Pitfall:** `datetime.strptime(r["Date"], "%Y-%m-%d")` is format-strict — a single `01/01/1981` row raises `ValueError` and stops the whole load. If you ever swap in a different sensor export, check one raw date string and adjust the format *before* debugging downstream maths. Also prefer `.date()` over leaving a full `datetime`: comparing a midnight timestamp to a plain date is a classic source of off-by-one day bugs.

In [ ]:
# Quick sanity checks (like Excel filters)
all_t = [t for _, t in temps]
print("min/max:", min(all_t), max(all_t))          # 0.0 26.3
print("mean:", round(statistics.mean(all_t), 3))   # 11.178
print("stdev:", round(statistics.pstdev(all_t), 3))# 4.071


**What to notice:**
- `min/max` print `0.0` and `26.3` — a full `26.3 °C` span between the coldest and hottest *minimum* temperatures in the decade; Exercise 1's follow-up asks for exactly this range.
- `mean` of `11.178` sits well below the midpoint of that range, reminding you that Melbourne overnight lows cluster cool rather than being uniform between 0 and 26.3.
- `stdev` uses `statistics.pstdev` (population) and rounds to `4.071`; that single number feeds the z-scores in Section 6, so if it drifts, your anomaly count of `170` will drift too.
- Printing these three stats first is the temperature equivalent of checking row count, min, max, and average before building any chart.

> **Pitfall:** `statistics.mean` / `pstdev` accept an iterable of numbers, not rows — always feed them `[t for _, t in temps]` (or a generator of temps). Passing `temps` itself would try to average `date` objects and `float`s together and raise `TypeError`.

---

## 2. Hottest / coldest day

Why it matters for analysts: "which day was extreme?" questions show up in every ops report, and `max(..., key=)` finds the whole row in one pass instead of sorting the table or writing a manual comparison loop.

In [ ]:
hottest = max(temps, key=lambda x: x[1])
coldest = min(temps, key=lambda x: x[1])
print("Hottest:", hottest[0], hottest[1])  # 1982-02-15 26.3
print("Coldest:", coldest[0], coldest[1])  # 1982-06-05 0.0


**What to notice:**
- Hottest day is `1982-02-15` at `26.3` and coldest is `1982-06-05` at `0.0` — note these are *minimum* temperatures, so `26.3` means the overnight low never dropped below 26.3 °C that day.
- Both results are full `(date, temp)` tuples: the `key=lambda x: x[1]` argument tells `max`/`min` to compare only the temperature element while returning the entire row, which is the Pythonic `=INDEX(MATCH(MAX(range)))`.
- February (summer in Melbourne) holding the hottest minimum and June (winter) holding `0.0` matches the seasonal pattern you'd expect — a quick domain sanity check on the parse.

> `max(seq, key=…)` is the Pythonic `=INDEX(MATCH(MAX(range)))`. Prefer it over manual loops when you only need one row.

> **Pitfall:** `max(temps)` **without** `key=` compares tuples element-by-element, so it would return the *latest date* (dates sort first), not the hottest day. Always pass `key=lambda x: x[1]` (or `itemgetter(1)`) when the ranking column is not the first element of the tuple.

---

## 3. Monthly averages with a dict of lists

Why it matters for analysts: monthly rollups are the bread-and-butter of sensor reporting — "average January 1981," "mean per calendar month across the decade" — and the pattern is always the same: bucket each row by a key, append its value, then average each bucket. In Excel that's a PivotTable; in pure Python it's a `defaultdict(list)`.

In [ ]:
by_month = defaultdict(list)          # key: (year, month) -> temps
for d, t in temps:
    by_month[(d.year, d.month)].append(t)

monthly_mean = {k: round(statistics.mean(v), 3)
                for k, v in sorted(by_month.items())}

print(len(monthly_mean), "months")                     # 120 months
print("1981-01 mean:", monthly_mean[(1981, 1)])        # 17.713
# Print first 6
for (y, m), mean in list(monthly_mean.items())[:6]:
    print(f"{y}-{m:02d}: {mean}")


Mental model: `defaultdict(list)` + `.append` = PivotTable “group rows by month, collect temps”, then one dict comprehension applies `statistics.mean` to every bucket.

**What to notice:**
- `120 months` = 10 years × 12 calendar months — if you saw 119 or 121, a date would have been misparsed into the wrong `(year, month)` bucket (e.g. month `0` or `13`).
- `1981-01 mean` is `17.713`, a January (summer) average far above the overall mean of `11.178` from Section 1 — seasonality is visible immediately once rows are grouped.
- Keys are `(year, month)` tuples, not just month numbers: `(1981, 1)` and `(1990, 1)` stay separate, so you can report either a specific month-year or average all Januaries later.
- The `sorted(by_month.items())` wrapper makes iteration chronological, so the printed first six lines walk `1981-01` forward rather than relying on insertion order.

> **Pitfall:** A plain `dict` would need `by_month.setdefault(key, []).append(t)` (or an `if key not in …` dance) to avoid `KeyError` on the first row of each month — `defaultdict(list)` creates the empty list for you. The reverse pitfall appears when *reading*: `by_month[(1981, 2)]` on a plain dict (or a misspelled key like `(1981, 2.0)`) raises `KeyError` instead of returning an average, so build keys consistently from `d.year, d.month`.

---

## 4. Count days above 25 °C

Why it matters for analysts: threshold counts ("how many days over X?") are the simplest anomaly filter and the same shape as an Excel filter column plus a COUNT. Writing it as a comprehension keeps both the matches and the count available for follow-up questions.

In [ ]:
hot_days = [(d, t) for d, t in temps if t > 25]
print("days > 25°C:", len(hot_days))  # 2
print(hot_days)
# [(datetime.date(1982, 1, 20), 25.2), (datetime.date(1982, 2, 15), 26.3)] — run it!


Only two scorchers in a decade of *minimum* temps — makes sense: these are overnight lows, so exceeding 25 °C even at night is genuinely rare.

**What to notice:**
- Exactly `2` days break `Temp > 25`: `1982-01-20` at `25.2` and `1982-02-15` at `26.3` — the second one is the same date as the decade's overall hottest day from Section 2, so the two analyses agree.
- The comparison uses a strict `>`, not `>=`: a day at exactly `25.0` would **not** count, which is worth stating explicitly if a stakeholder asks "25 or above?".
- Both matches are kept as full `(date, temp)` tuples, so you can print the list for audit rather than only reporting a bare integer.
- Over 3650 days, 2 hits is `2/3650 ≈ 0.0005` — a useful reminder that mean-min temps and heatwave days are very different questions on the same column.

> **Pitfall:** Threshold direction matters on sorted-looking data: `t > 25` excludes `25.0`, while `t >= 25` includes it. Also, don't compare temperatures as strings — `"9.5" > "10.2"` is `True` lexicographically; the loader's `float(r["Temp"])` is what makes this filter trustworthy.

---

## 5. Longest run of consecutive days below 10 °C

Why it matters for analysts: streak and run detection ("longest cold snap," "consecutive days breached") cannot be done with a single filter — you must walk the series in calendar order and remember context from the previous row. The `prev_cold` / `prev_date` variables below are the standard state you'll reuse for any consecutive-day metric.

Streak logic: walk the series in order; if today is cold **and** yesterday was the previous calendar day, extend the run; else start a new run.

In [ ]:
best_len = cur_len = 0
best_start = best_end = None
cur_start = None
prev_date = None
prev_cold = False

for d, t in temps:
    if t < 10:
        # extend only if previous calendar day was also cold
        if prev_cold and prev_date is not None and (d - prev_date).days == 1:
            cur_len += 1
        else:
            cur_len = 1
            cur_start = d
        if cur_len > best_len:
            best_len = cur_len
            best_start, best_end = cur_start, d
        prev_cold = True
    else:
        cur_len, cur_start = 0, None
        prev_cold = False
    prev_date = d

print(f"Longest cold streak: {best_len} days ({best_start} → {best_end})")
# Longest cold streak: 71 days (1982-05-31 → 1982-08-09)


**What to notice:**
- The winner is `71 days` from `1982-05-31 → 1982-08-09` — spanning the southern-hemisphere winter (June–August), which is exactly when you'd expect a sub-10 °C run of that length.
- `best_start` / `best_end` are only updated when `cur_len > best_len`, so ties keep the **first** longest streak found; a strict `>=` would instead slide the window to the last tie.
- `prev_cold` starts as `False`, so the very first cold day always begins a fresh run of `1` even if it happens to be the file's first row — no phantom run length of `0` or double-count.
- On a warm day (`t >= 10`) both `cur_len` and `prev_cold` reset, but `prev_date` still updates: the next iteration can correctly measure a gap even across a warm period.

> Why `.days == 1` matters: without the date check, any two cold days (even weeks apart) would merge into one “streak”.

> **Pitfall:** Two guards must both pass before extending a run: `prev_cold` (yesterday qualified) **and** `(d - prev_date).days == 1` (yesterday was the actual previous calendar day). Dropping either one fails differently — without `prev_cold`, a warm yesterday followed by a cold day with a 1-day gap would wrongly extend; without the `.days == 1` check, a missing date or a two-day hole would glue separate cold snaps together into a fake record streak.

---

## 6. Anomaly flagging: |z| > 2

Why it matters for analysts: z-scores turn raw readings into "how unusual is this?" so one threshold works across sensors with different means and spreads. Flagging `|z| > 2` is the pure-Python version of an conditional-format rule on a stats sheet — worth a second look in ops dashboards without hand-picking cut-offs per month.

In [ ]:
mu = statistics.mean(all_t)
sd = statistics.pstdev(all_t)
anomalies = [(d, t, round((t - mu) / sd, 2))
             for d, t in temps
             if abs((t - mu) / sd) > 2]

print(f"{len(anomalies)} days with |z| > 2")  # 170
print("sample:", anomalies[:5])


A z-score is just “how many standard deviations from the mean.” |z| > 2 ≈ the tails of a normal-ish distribution — worth a second look in ops dashboards.

**What to notice:**
- `170` of `3650` days trip the rule — about `0.0466` of the decade, which Exercise 3's follow-up checks against the ~4.6% a normal distribution predicts beyond 2 SD, so the empirical tails line up closely with theory here.
- Each anomaly is stored as `(date, temp, rounded_z)`: keeping the z value beside the raw temp lets a reviewer see *how* extreme a point is, not just that it crossed the line.
- The list comprehension uses `mu` and `sd` computed **once** from the full series; recomputing them inside the loop per row would be slower and could subtly change results if you ever switched to a rolling window by mistake.
- Because `abs(...)` is applied, both unusually cold (`z < -2`) and unusually warm (`z > +2`) days appear — the `0.0 °C` June day and the `26.3 °C` February day are both candidates for the tail.

> **Pitfall:** Section 6 uses `statistics.pstdev` (population SD), while the Exercise 3 solution recomputes with `statistics.stdev` (sample SD). With n = 3650 the two are nearly identical and both still report `170` flagged days here — but if you switch datasets or small samples, pick one SD definition and use it consistently, or your z-scores and thresholds will quietly disagree between cells.

---

## Exercises (do these!)

### Exercise 1 — Monthly mean temps dict
Build `monthly_mean` as above (you may already have it). Print the mean for **1981-01** and for **1990-12**.
*Expected: 1981-01 ≈ 17.713; 1990-12 ≈ 14.368 (3 s.f. may differ slightly by rounding).*

**Follow-up:** What is the full temperature range over the decade? Check: 26.3 C.

<details>
<summary>Hint</summary>

```python
by_month = defaultdict(list)
for d, t in temps:
    by_month[(d.year, d.month)].append(t)
monthly_mean = {k: statistics.mean(v) for k, v in by_month.items()}
print(monthly_mean[(1981, 1)], monthly_mean[(1990, 12)])
```

</details>

### Exercise 2 — Count days Temp > 25
How many days had `Temp > 25`? Print the count and each `(date, temp)` pair.
*Expected: 2 days — 1982-01-20 (25.2) and 1982-02-15 (26.3).*

**Follow-up:** What fraction of 1982 (non-leap, 365 days) did the cold streak cover? Check: about 0.1945.

<details>
<summary>Hint</summary>

List comprehension + `len()`: `hot = [(d,t) for d,t in temps if t > 25]`.
</details>

### Exercise 3 — Longest run Temp < 10
Find the longest **consecutive-calendar-day** run where `Temp < 10`. Print length and start/end dates.
*Expected: 71 days, 1982-05-31 → 1982-08-09.*

**Follow-up:** What share of days are anomalies, and how does that compare with the roughly 4.6 percent a normal distribution predicts beyond 2 SD? Check: 170 days, about 0.0466.

<details>
<summary>Hint</summary>

Reuse the streak walker from Section 5; the gap check is `(d - prev_date).days == 1`.
</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
from collections import defaultdict
import statistics
by_month = defaultdict(list)
for d, t in temps:
    by_month[(d.year, d.month)].append(t)
monthly_mean = {k: statistics.mean(v) for k, v in by_month.items()}
print(f"1981-01: {monthly_mean[(1981, 1)]:.3f}")   # 17.713
print(f"1990-12: {monthly_mean[(1990, 12)]:.3f}")  # ~14.368

# --- Solution 2 ---
hot = [(d, t) for d, t in temps if t > 25]
print("count:", len(hot))  # 2
for d, t in hot:
    print(d, t)
# 1982-01-20 25.2
# 1982-02-15 26.3

# --- Solution 3 ---
best_len = cur_len = 0
best_start = best_end = cur_start = prev_date = None
prev_cold = False
for d, t in temps:
    if t < 10:
        if prev_cold and prev_date is not None and (d - prev_date).days == 1:
            cur_len += 1
        else:
            cur_len, cur_start = 1, d
        if cur_len > best_len:
            best_len = cur_len
            best_start, best_end = cur_start, d
        prev_cold = True
    else:
        cur_len, cur_start = 0, None
        prev_cold = False
    prev_date = d
print(f"{best_len} days: {best_start} → {best_end}")
# 71 days: 1982-05-31 → 1982-08-09

# --- Follow-up 1 ---
tr = round(max(t for _, t in temps) - min(t for _, t in temps), 1)
print(tr)  # 26.3
assert tr == 26.3

# --- Follow-up 2 ---
cov = round(best_len / 365, 4)
print(cov)  # ~0.1945
assert cov == 0.1945

# --- Follow-up 3 ---
import statistics as _st
_mu = _st.mean(t for _, t in temps)
_sd = _st.stdev(t for _, t in temps)
_out = [d for d, t in temps if abs((t - _mu) / _sd) > 2]
shr = round(len(_out) / len(temps), 4)
print(len(_out), shr)  # 170 0.0466
assert len(_out) == 170 and shr == 0.0466


### What to learn next
- Rolling means: implement a 7-day window with `collections.deque(maxlen=7)`.
- Plotting: hand this list to `matplotlib.pyplot.plot(dates, temps)`.
- Then pandas: `pd.read_csv(..., parse_dates=["Date"]).set_index("Date").resample("M").mean()` does Sections 3–4 in two lines.
- Cheat sheet: `max(…, key=)` → extreme row, `defaultdict(list)` → group-by, streak counter → `(d - prev).days == 1`, z-score → `(x - μ) / σ`.

*Files: `datasets/daily-min-temperatures.csv` (local) · raw GitHub fallback for Colab in `load_temps()`. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
